# Visualizing Cycle Representatives

Start here if you want to see what a `PersistenceForest` contains. This notebook builds one small 2D forest and one small 3D forest, then shows the plotting calls you will use most often:

- `plot_barcode`
- `plot_at_filtration`
- `plot_barcode_cycle_reps`
- `plot_filtration_interactive`
- `plot_at_filtration_plotly`

The examples keep the point clouds small so the plots render quickly.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from persforest import PersistenceForest


def sample_noisy_circle(n=120, noise=0.035, seed=4):
    rng = np.random.default_rng(seed)
    theta = np.linspace(0.0, 2.0 * np.pi, n, endpoint=False)
    theta = theta + rng.normal(scale=0.01, size=n)
    radius = 1.0 + rng.normal(scale=noise, size=n)
    return np.column_stack((radius * np.cos(theta), radius * np.sin(theta)))


def sample_noisy_sphere(n=90, noise=0.04, seed=8):
    rng = np.random.default_rng(seed)
    z = rng.uniform(-1.0, 1.0, n)
    theta = rng.uniform(0.0, 2.0 * np.pi, n)
    radius = 1.0 + rng.normal(scale=noise, size=n)
    xy = np.sqrt(1.0 - z * z)
    sphere = np.column_stack((xy * np.cos(theta), xy * np.sin(theta), z))
    return radius[:, None] * sphere


## Build a 2D forest

The noisy circle has one dominant 1-dimensional feature. `min_bar_length=0.05` selects that long bar and ignores short noise bars.


In [ ]:
points_2d = sample_noisy_circle()
forest_2d = PersistenceForest(points_2d)

fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(points_2d[:, 0], points_2d[:, 1], s=8, color="black")
ax.set_aspect("equal")
ax.set_title("2D point cloud")
plt.show()


## Barcode and active cycles

`plot_barcode` shows the lifetime of each bar. `plot_at_filtration` shows the alpha-complex and the cycle representatives active at one filtration value.


In [ ]:
forest_2d.plot_barcode(min_bar_length=0.01, coloring="forest", bar_width=3)

forest_2d.plot_at_filtration(
    filt_val=0.45,
    min_bar_length=0.05,
    coloring="forest",
    vertex_size=8,
    show=True,
)


## One representative per long bar

`plot_barcode_cycle_reps` samples each selected bar at `birth + relative_position * lifespan`. The same convention is used by `barcode_cycle_reps` when you extract representatives as data.


In [ ]:
forest_2d.plot_barcode_cycle_reps(
    relative_position=0.1,
    min_bar_length=0.05,
    coloring="bars",
    linewidth_cycle=2.0,
    vertex_size=8,
    style_2d={"point_color": "0.15", "point_alpha": 0.75},
)


## Interactive filtration

`plot_filtration_interactive` returns a Plotly figure with a slider over filtration values. Display the returned figure in a notebook to inspect the filtration.


In [ ]:
interactive_2d = forest_2d.plot_filtration_interactive(
    filt_max=0.8,
    min_bar_length=0.05,
    coloring="bars",
    resolution=24,
    vertex_size=4,
    height=550,
    show=False,
)

interactive_2d


## Signed and unsigned display

By default, plotting cancels opposite-oriented duplicate simplices before drawing. Set `signed=True` to preserve the signed chain, and use orientation arrows in 2D when orientation matters.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

forest_2d.plot_at_filtration(
    0.28,
    ax=axes[0],
    signed=False,
    min_bar_length=0.05,
    coloring="bars",
    title="unsigned display",
    show=False,
)

forest_2d.plot_at_filtration(
    0.28,
    ax=axes[1],
    signed=True,
    min_bar_length=0.05,
    coloring="bars",
    title="signed display",
    show=False,
    style_2d={"show_orientation_arrows": True, "cycle_edge_width": 2.0},
)

plt.tight_layout()
plt.show()


## Build a 3D forest

For a 3D point cloud, codimension-one cycle representatives are triangular surface chains. Plotly is the most convenient backend for inspecting them interactively.


In [ ]:
points_3d = sample_noisy_sphere()
forest_3d = PersistenceForest(points_3d)

long_bars = sorted(forest_3d.barcode, key=lambda bar: bar.lifespan(), reverse=True)
[(bar.birth, bar.death, bar.lifespan()) for bar in long_bars[:3]]


## 3D Plotly snapshot

`show=False` builds the Plotly figure without opening a renderer. In a notebook, display the variable `fig_3d` to inspect the scene.


In [ ]:
fig_3d = forest_3d.plot_at_filtration_plotly(
    filt_val=1.5,
    min_bar_length=0.1,
    coloring="forest",
    show_complex=True,
    show_cycles=True,
    vertex_size=2,
    cycle_opacity=0.85,
    height=650,
    show=False,
)

fig_3d


## 3D filtration slider

Use a small `resolution` while experimenting, then increase it for presentation-quality output.


In [ ]:
slider_3d = forest_3d.plot_filtration_interactive(
    filt_max=2,
    min_bar_length=0.1,
    coloring="forest",
    show_complex=True,
    show_cycles=True,
    vertex_size=2,
    cycle_opacity=0.75,
    resolution=20,
    height=650,
    show=False,
)

slider_3d
